# 00 — Target Data Preparation

**Private notebook — do not share with students.**

Embeds the target sentences and saves them as a FAISS index + parquet.  
Students only receive the output files, not this notebook.

**Input**  : `data/target_sentences.txt`  
**Output** : `data/sentences_target_text_db.parquet` + `data/sentences_target_vector_db.index`

In [ ]:
from pathlib import Path

Path("data/sentences_target_text_db.parquet").unlink(missing_ok=True)
Path("data/sentences_target_vector_db.index").unlink(missing_ok=True)

In [ ]:
import faiss
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

TARGET_FILE        = 'data/target_sentences.txt'
TARGET_OUTPUT_FILE = 'data/sentences_target_text_db.parquet'
TARGET_INDEX_FILE  = 'data/sentences_target_vector_db.index'
MODEL_NAME         = 'all-mpnet-base-v2'

In [ ]:
target_records = []
for line in Path(TARGET_FILE).read_text().splitlines():
    if ':' in line:
        tid, sentence = line.split(':', 1)
        target_records.append({'target_id': tid.strip(), 'text': sentence.strip()})

target_df = pd.DataFrame(target_records)
target_df.insert(0, 'id', range(len(target_df)))

In [ ]:
model = SentenceTransformer(MODEL_NAME)

In [ ]:
target_emb = model.encode(
    target_df['text'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

In [ ]:
target_index = faiss.IndexFlatIP(target_emb.shape[1])
target_index.add(target_emb)

target_df.to_parquet(TARGET_OUTPUT_FILE, index=False)
faiss.write_index(target_index, TARGET_INDEX_FILE)
print(f'Saved {len(target_df)} target rows → {TARGET_OUTPUT_FILE}')
print(f'Saved FAISS index → {TARGET_INDEX_FILE} ({target_index.ntotal} vectors, dim={target_index.d})')
print()
for _, row in target_df.iterrows():
    print(f'{row["target_id"]}: {row["text"]}')